In [1]:
import warnings
warnings.filterwarnings('ignore')

import evaluate
import pandas as pd
import re
import json
import mlflow
from datasets import load_dataset, Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, pipeline

## Data Preparation

In [2]:
# Loading the dataset
df = load_dataset("Tobi-Bueck/customer-support-tickets")['train'].to_pandas()

# Cleaning the text data
def clean_text(text):
    if pd.isna(text):
        return ""

    text = re.sub(r'\\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()  # Convert all text to lowercase

    return text

df['cleaned_body'] = df['body'].apply(clean_text)
df['cleaned_subject'] = df['subject'].apply(clean_text)

# Combining subject and body for input

def combine_subject_body(row):
    subject = row['cleaned_subject']
    body = row['cleaned_body']
    if subject:
        return subject + '. ' + body
    else:
        return body


df['subject_body'] = df.apply(combine_subject_body, axis=1)

department_mapping = {
    # Support Engineering
    "Technical Support": "Support Engineering",
    "Product Support": "Support Engineering",
    "IT & Technology/Hardware Support": "Support Engineering",
    "IT & Technology/Software Development": "Support Engineering",
    "IT & Technology/Network Infrastructure": "Support Engineering",
    "IT & Technology/Security Operations": "Support Engineering",

    # Internal IT
    "IT Support": "Internal IT",

    # Operations
    "Service Outages and Maintenance": "Operations",

    # Finance
    "Billing and Payments": "Finance",

    # Sales
    "Sales and Pre-Sales": "Sales",

    # Customer Operations
    "Returns and Exchanges": "Customer Operations",
    "Customer Service": "Customer Operations",
    "General Inquiry": "Customer Operations",

    # Human Resources
    "Human Resources": "Human Resources",
    "Jobs & Education/Recruitment": "Human Resources",
}

df["department"] = (
    df["queue"]
      .map(department_mapping)
      .fillna("Others")
)

df = df[df['department'] != 'Others']
df = df[['subject_body', 'department']]
df


,subject_body,department
0,wesentlicher sicherheitsvorfall. sehr geehrtes...,Support Engineering
1,account disruption. dear customer support team...,Support Engineering
2,query about smart home system integration feat...,Customer Operations
3,inquiry regarding invoice details. dear custom...,Finance
4,question about marketing agency software compa...,Sales
...,...,...
61760,assistance needed for ifttt docker integration...,Support Engineering
61761,bitten um unterstützung bei der integration. s...,Support Engineering
61762,"hello customer support, i am inquiring about t...",Finance
61763,hilfe bei digitalen strategie-problemen. die q...,Support Engineering


### Converting to Hugginface Dataset

In [3]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset, ClassLabel

le = LabelEncoder()

df['label'] = le.fit_transform(df['department'])

print("Class labels")
for i, name in enumerate(le.classes_):
    print(i, "->", name)

class_label = ClassLabel(names=list(le.classes_))

# First split: 80% train, 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# Second split: split the remaining 20% into 10% validation and 10% test
eval_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

# Reset indices
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
eval_dataset = Dataset.from_pandas(eval_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)


train_dataset = train_dataset.cast_column("label", class_label)
eval_dataset = eval_dataset.cast_column("label", class_label)
test_dataset = test_dataset.cast_column("label", class_label)

Class labels
0 -> Customer Operations
1 -> Finance
2 -> Human Resources
3 -> Internal IT
4 -> Operations
5 -> Sales
6 -> Support Engineering


Casting the dataset: 100%|██████████| 5010/5010 [00:00<00:00, 1424834.76 examples/s]


### Tokenizer for Bert Model

In [4]:
from transformers import AutoTokenizer
model_checkpoint = 'distilbert-base-multilingual-cased'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [5]:
text = "trotz der umsetzung unserer standardisierten behebungs- und"
inputs = tokenizer(text, return_tensors="pt")

print("Tokenized Input:")
for k, v in inputs.items():
    print(f"{k}: {v.shape} -> {v}")

Tokenized Input:
input_ids: torch.Size([1, 16]) -> tensor([[   101,  43664,  10118,  97615, 102993,  15826,  22981,  14979,  79219,
          10115,  10347,  98952,  10107,    118,  10130,    102]])
token_type_ids: torch.Size([1, 16]) -> tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
attention_mask: torch.Size([1, 16]) -> tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [6]:
def preprocess_function(examples):
    return tokenizer(examples['subject_body'], truncation=True, padding='max_length')

train_dataset = train_dataset.map(preprocess_function, batched=True)
eval_dataset = eval_dataset.map(preprocess_function, batched=True)



Map: 100%|██████████| 5010/5010 [00:00<00:00, 5554.09 examples/s]


### Model

In [9]:
from transformers import AutoModelForSequenceClassification

num_labels = len(le.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
)

# Freeze DistilBERT encoder
for param in model.distilbert.parameters():
    param.requires_grad = False

# Train only the classification head
for param in model.pre_classifier.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8311.31it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "Total": total_params,
        "Trainable": trainable_params,
        "Frozen": total_params - trainable_params
    }

# Example
param_counts = count_parameters(model)
print("Model Parameter Counts:")
for k, v in param_counts.items():
    print(f'{k}: {v:,}')


Model Parameter Counts:
Total: 135,330,055
Trainable: 595,975
Frozen: 134,734,080


In [11]:
outputs = model(**inputs)
print(outputs)

SequenceClassifierOutput(loss=None, logits=tensor([[-0.0824,  0.0158,  0.0133,  0.0355,  0.0508,  0.1801,  0.0060]],
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


In [12]:
import evaluate
import numpy as np

accuracy_metrics = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return accuracy_metrics.compute(predictions=predictions, references=labels)

In [13]:
# Defining training arguments

training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=10,
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

In [14]:
# Initialize trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.406924,1.398778,0.491417
2,1.409636,1.374720,0.504391
3,1.388102,1.369457,0.505589


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.77s/it]


TrainOutput(global_step=7515, training_loss=1.4084048193133996, metrics={'train_runtime': 1343.6705, 'train_samples_per_second': 89.484, 'train_steps_per_second': 5.593, 'total_flos': 1.5928902832407552e+16, 'train_loss': 1.4084048193133996, 'epoch': 3.0})

In [16]:
label_map = train_dataset.features['label'].int2str

In [17]:
import torch

def classify(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs = {k:v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class_id = outputs.logits.argmax(dim=-1).item()
    return label_map(predicted_class_id)

In [18]:
text = "Give me the recent updates on the product"
preprocessed_text = clean_text(text)
print(preprocessed_text)
print(classify(preprocessed_text))

give me the recent updates on the product
Support Engineering
